# CrewAI Multi-Agent Content Workflow

## Research → article → social media, reconstructed for Google Colab

This notebook reconstructs the workflow shown in the supplied screenshots and organizes it as a concise, executable teaching artifact. Three specialized agents collaborate through explicit task hand-offs:

1. a **Researcher** gathers five credible insights;
2. a **Writer** turns those notes into a structured article; and
3. a **Social Media Strategist** adapts the article for LinkedIn and X/Twitter.

The default configuration is safe: it validates and previews the workflow without requiring an API key or publishing anything. Live LLM execution is an intentional opt-in later in the notebook.

## Learning objectives and design interpretation

CrewAI models a crew as agents with distinct roles, tasks with explicit output contracts, and a process that coordinates execution. This example uses a sequential process because each stage depends on the preceding stage. That makes the information flow inspectable and avoids multiple agents independently repeating the same work.

```text
Topic → Researcher → research notes → Writer → article → Social Strategist → channel-ready posts
```

The AILaunchpad material emphasizes the same production principles used here: clear delegation, shared context, completion criteria, least-privilege tools, explicit human approval before external actions, and observable execution.

## 1. Install dependencies

Run this cell once in a fresh Colab runtime. `%pip` installs into the active notebook kernel. If Colab asks for a restart after upgrades, restart the session and continue from the imports cell.

In [ ]:
%pip install -q -U crewai langchain-openai requests pydantic

## 2. Imports and runtime configuration

`ENABLE_LIVE_RUN=False` is deliberate. It lets every setup and validation cell run without a key and prevents accidental API usage. When you are ready, store `OPENAI_API_KEY` in Colab **Secrets** (key icon), enable notebook access, and opt in near the end. The optional webhook remains disabled unless both a URL and an explicit publish flag are supplied.

In [ ]:
import os
from pathlib import Path
from typing import Optional, Type

import requests
from pydantic import BaseModel, Field
from crewai import Agent, Crew, Process, Task
from crewai.tools import BaseTool
from langchain_openai import ChatOpenAI

ENABLE_LIVE_RUN = False
ALLOW_EXTERNAL_PUBLISH = False
MODEL_NAME = "gpt-4o-mini"
TOPIC = "State of Canada's economic growth in the 21st century"

def load_colab_secret(name: str) -> Optional[str]:
    """Read a secret from Colab when available, otherwise from the environment."""
    value = os.getenv(name)
    if value:
        return value
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name)
    except Exception as exc:
        # Missing/denied Colab secrets should keep safe-preview mode executable.
        if exc.__class__.__module__.startswith("google.colab") or isinstance(exc, ImportError):
            return None
        raise

OPENAI_API_KEY = load_colab_secret("OPENAI_API_KEY")
SOCIAL_WEBHOOK_URL = load_colab_secret("SOCIAL_WEBHOOK_URL")
print("Mode:", "LIVE" if ENABLE_LIVE_RUN else "SAFE PREVIEW")
print("OpenAI key available:", bool(OPENAI_API_KEY))
print("Webhook configured:", bool(SOCIAL_WEBHOOK_URL))

## 3. Helper implementations and CrewAI tools

The original notebook exposed two capabilities to the social agent: generating channel-specific copy and optionally sending it to a webhook. Here they are expressed as typed `BaseTool` classes. Publishing uses two gates—configuration plus explicit approval—so merely executing the notebook cannot create an external side effect.

In [ ]:
def _make_social_copy_impl(
    title: str, summary: str, platform: str = "linkedin"
) -> str:
    """Use the configured LLM to create concise social copy."""
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is required for live copy generation.")
    llm = ChatOpenAI(model=MODEL_NAME, api_key=OPENAI_API_KEY, temperature=0.3)
    prompt = (
        f"Craft a concise {platform} post promoting an article.\n"
        f"Title: {title}\nSummary: {summary}\n"
        "Constraints: 1-2 punchy lines, a clear CTA, and 3-5 relevant hashtags."
    )
    return str(llm.invoke(prompt).content)

def _post_via_webhook_impl(text: str, image_path: Optional[str] = None) -> str:
    """Publish only when a human has explicitly enabled the external action."""
    if not ALLOW_EXTERNAL_PUBLISH:
        return "Not published: ALLOW_EXTERNAL_PUBLISH is False."
    if not SOCIAL_WEBHOOK_URL:
        raise RuntimeError("SOCIAL_WEBHOOK_URL is not configured.")
    if image_path:
        path = Path(image_path)
        if not path.is_file():
            raise FileNotFoundError(path)
        with path.open("rb") as image_file:
            response = requests.post(
                SOCIAL_WEBHOOK_URL, data={"text": text},
                files={"image": image_file}, timeout=60
            )
    else:
        response = requests.post(
            SOCIAL_WEBHOOK_URL, data={"text": text}, timeout=60
        )
    response.raise_for_status()
    return "Posted"

In [ ]:
class MakeSocialCopyInput(BaseModel):
    title: str = Field(description="Article title")
    summary: str = Field(description="Short article summary")
    platform: str = Field(default="linkedin", description="Target platform")

class MakeSocialCopyTool(BaseTool):
    name: str = "make_social_copy"
    description: str = "Create a short, platform-optimized social post."
    args_schema: Type[BaseModel] = MakeSocialCopyInput

    def _run(self, title: str, summary: str, platform: str = "linkedin") -> str:
        return _make_social_copy_impl(title, summary, platform)

class PostViaWebhookInput(BaseModel):
    text: str = Field(description="Post text")
    image_path: Optional[str] = Field(default=None, description="Optional local image path")

class PostViaWebhookTool(BaseTool):
    name: str = "post_via_webhook"
    description: str = "Publish a post through the configured webhook after explicit approval."
    args_schema: Type[BaseModel] = PostViaWebhookInput

    def _run(self, text: str, image_path: Optional[str] = None) -> str:
        return _post_via_webhook_impl(text, image_path)

make_social_copy = MakeSocialCopyTool()
post_via_webhook = PostViaWebhookTool()
print("Tools ready:", make_social_copy.name, "and", post_via_webhook.name)

## 4. Define the agents

Roles and goals act as behavioral boundaries. Delegation is disabled because the orchestration is already explicit: each agent owns one stage and the crew passes outputs through task context. The LLM is created only inside the factory, so safe-preview mode does not attempt authentication.

In [ ]:
def build_agents() -> tuple[Agent, Agent, Agent]:
    if not OPENAI_API_KEY:
        raise RuntimeError("Add OPENAI_API_KEY to Colab Secrets before a live run.")
    llm = ChatOpenAI(model=MODEL_NAME, api_key=OPENAI_API_KEY, temperature=0.2)

    researcher = Agent(
        role="Researcher",
        goal="Collect five up-to-date, credible insights on the topic.",
        backstory="A careful analyst who synthesizes evidence and cites sources.",
        allow_delegation=False,
        llm=llm,
    )
    writer = Agent(
        role="Writer",
        goal="Draft a clear 250-300 word article grounded in the research notes.",
        backstory="A senior technical writer who preserves evidence and structure.",
        allow_delegation=False,
        llm=llm,
    )
    social_agent = Agent(
        role="Social Media Strategist",
        goal="Adapt the article into platform-appropriate posts without publishing.",
        backstory="A channel strategist who understands tone, length, CTAs, and hashtags.",
        tools=[make_social_copy],
        allow_delegation=False,
        llm=llm,
    )
    return researcher, writer, social_agent

## 5. Define tasks and context hand-offs

Each `expected_output` is a lightweight completion contract. `context=[research_task]` gives the Writer the upstream research; the social task receives both research and article context. The research prompt explicitly requests verifiable links and warns against inventing citations—an important correction because observed demo output contained claims that should not be treated as validated facts.

In [ ]:
def build_tasks(
    researcher: Agent, writer: Agent, social_agent: Agent, topic: str
) -> tuple[Task, Task, Task]:
    research_task = Task(
        description=(
            f"Research '{topic}'. Find five key insights covering current developments, "
            "challenges, and opportunities. Include a source name and URL for every insight. "
            "Do not invent facts, dates, statistics, or citations; flag uncertainty explicitly."
        ),
        agent=researcher,
        expected_output="Five evidence-based bullets, each with a short interpretation and source URL.",
    )
    writing_task = Task(
        description=(
            "Write a structured 250-300 word article from the research notes. Include a "
            "compelling title, summary, introduction, 3-4 short sections, and conclusion. "
            "Preserve source links and do not add unsupported claims."
        ),
        agent=writer,
        expected_output="A complete Markdown article with title, summary, sections, conclusion, and sources.",
        context=[research_task],
    )
    social_task = Task(
        description=(
            "Create one LinkedIn post and one X/Twitter post from the article. Use "
            "make_social_copy for each platform. Show the drafts only—do not publish. "
            "Extract the title and summary from the article context."
        ),
        agent=social_agent,
        expected_output="LinkedIn draft, X/Twitter draft, and publishing status: not published.",
        context=[research_task, writing_task],
    )
    return research_task, writing_task, social_task

## 6. Build the crew

Sequential execution is the natural fit for this dependency chain. Verbose tracing is enabled for learning and debugging. In a production system, attach durable tracing, redact secrets and sensitive content, record costs/latency, and define retry and timeout policies.

In [ ]:
def build_crew(topic: str = TOPIC) -> Crew:
    agents = build_agents()
    tasks = build_tasks(*agents, topic=topic)
    return Crew(
        agents=list(agents),
        tasks=list(tasks),
        process=Process.sequential,
        verbose=True,
    )

## 7. Key-free preview and structural checks

This cell is the default executable endpoint. It confirms the intended orchestration and external-action guard without calling a model. It is not a substitute for a live integration test, but it catches configuration drift and makes the notebook safe to run top-to-bottom before keys are integrated.

In [ ]:
workflow_contract = {
    "topic": TOPIC,
    "process": "sequential",
    "agents": ["Researcher", "Writer", "Social Media Strategist"],
    "handoffs": ["research → article", "research + article → social drafts"],
    "external_publish_enabled": ALLOW_EXTERNAL_PUBLISH,
}
assert len(workflow_contract["agents"]) == 3
assert workflow_contract["process"] == "sequential"
assert _post_via_webhook_impl("preview only").startswith("Not published")
workflow_contract

## 8. Optional live execution

To run the actual crew: add `OPENAI_API_KEY` to Colab Secrets, change `ENABLE_LIVE_RUN` to `True` in the configuration cell, and rerun from that cell onward. Publishing remains off. Live research quality depends on the model's available tools; for production-grade freshness, attach an approved search/retrieval tool and independently verify every cited source.

In [ ]:
if ENABLE_LIVE_RUN:
    if not OPENAI_API_KEY:
        raise RuntimeError("Live mode requires OPENAI_API_KEY in Colab Secrets.")
    crew = build_crew()
    result = crew.kickoff()
    print("\nWORKFLOW COMPLETED\n")
    print(result)
else:
    print("Safe preview complete. No LLM call or external publish was attempted.")

## 9. How to interpret and extend the workflow

A successful run should produce five sourced research insights, a short article grounded in those insights, and two visibly different platform drafts. Inspect the trace for whether each agent stayed within role, whether context flowed without unsupported additions, and whether the social agent avoided publishing.

Before production use, add: (1) a vetted search tool for genuine current research, (2) citation validation, (3) structured output schemas, (4) retries/timeouts and budget limits, (5) persistent tracing and evaluation cases, and (6) a human approval gate before any webhook is enabled. These additions map directly to the AILaunchpad themes of shared state, guardrails, observability, failure recovery, and cost control.

---
### Reconstruction notes

The screenshots contained overlapping views of the same Colab notebook and its execution trace. Repeated trace fragments were deduplicated; source code, role definitions, task flow, and safety intent were retained. Minimal additions were made for notebook executability without keys: delayed live-client construction, a safe preview path, typed tool inputs, explicit sequential processing, citation-quality instructions, and a two-gate publishing safeguard.